# DCOPF - Example 6 (PTDF with a Subset of Lines)

**Original AMPL author:** Xingpeng Li - Associate Professor, Dept. of Electrical and Computer Engineering, University of Houston (UH), Houston, TX, USA (Senior Member, IEEE). Email: xli83@central.uh.edu

**Converted by:** Haoxiang Wan - PhD student of Dr. Xingpeng Li (AMPL -> Pyomo + Gurobi)

Assumes lines 2 and 3 cannot overload, so their flow equations and thermal limits are dropped. Only line 1's PTDF expression and limit are retained. PTDF's per-line independence allows this simplification, which is **not** possible in B-theta.


In [1]:
from pyomo.environ import (
    ConcreteModel, Var, Objective, Constraint, SolverFactory,
    minimize, value
)

c1, Pgmin1, Pgmax1 = 10, 20, 70
c3, Pgmin3, Pgmax3 = 20, 40, 90
Load2 = 100
branchRate1 = 60

model = ConcreteModel()
model.G1 = Var(bounds=(Pgmin1, Pgmax1))
model.G3 = Var(bounds=(Pgmin3, Pgmax3))
model.pk1 = Var(bounds=(-branchRate1, branchRate1))

model.obj = Objective(expr=c1*model.G1 + c3*model.G3, sense=minimize)

model.lineFlow_1          = Constraint(expr=model.pk1 == 0.25*model.G1 + 0.5*Load2)
model.SysWidePowerBalance_1 = Constraint(expr=model.G1 + model.G3 == Load2)

In [2]:
# ---- Solve with Gurobi ----
solver = SolverFactory('gurobi')
solver.options['MIPGap'] = 0.0
solver.options['TimeLimit'] = 90
results = solver.solve(model, tee=True)
print(results.solver.status, results.solver.termination_condition)
print(f"G1 = {value(model.G1):.4f}, G3 = {value(model.G3):.4f}")
print(f"pk1 = {value(model.pk1):.4f}")

Read LP format model from file C:\Users\hwan6\AppData\Local\Temp\tmpy_f3blbw.pyomo.lp


Reading time = 0.00 seconds
x1: 2 rows, 3 columns, 4 nonzeros
Set parameter MIPGap to value 0
Set parameter TimeLimit to value 90
Gurobi Optimizer version 12.0.3 build v12.0.3rc0 (win64 - Windows 11.0 (26100.2))

CPU model: 12th Gen Intel(R) Core(TM) i7-12700, instruction set [SSE2|AVX|AVX2]


Thread count: 12 physical cores, 20 logical processors, using up to 20 threads

Non-default parameters:


TimeLimit  90


MIPGap  0



...

  Matrix range     [3e-01, 1e+00]


  Objective range  [1e+01, 2e+01]
  Bounds range     [2e+01, 9e+01]
  RHS range        [5e+01, 1e+02]


Presolve removed 2 rows and 3 columns
Presolve time: 0.00s
Presolve: All rows and columns removed


Iteration    Objective       Primal Inf.    Dual Inf.      Time
       0    1.6000000e+03   0.000000e+00   0.000000e+00      0s

Solved in 0 iterations and 0.00 seconds (0.00 work units)
Optimal objective  1.600000000e+03


ok optimal
G1 = 40.0000, G3 = 60.0000
pk1 = 60.0000
